# Retrieve & Re-Ranking LAB

Podstawą wszystkich rozwiązań typu RAG (ang. Retrieval Augmented Generation) jest komponent wyszukiwania. Wyszukiwanie polega na identyfikacji odpowiednich dokumentów, w naszym RAGOwym przypadku takich dokumentów, które potencjalnie zawierają informacje kluczowe do odpowiedzenia na postawione pytanie. Definicja dokumentu jest też rzeczą płynną, raz to może być paragraf, a raz strona, czy rozdział, zależy to od zadania, oraz modeli jakie używamy.

Na wykładzie starałem się wyraźnie zwrócić uwagę na to, że wyszukiwanie może być leksykalne (BM25) lub semantyczne (najczęściej oparte na głębokich sieciach neuronowych, używanych jako modeli embedderów tekstów lub ocaniejących dopasowanie pary tekst-pytanie).
Podczas wykładu dokładnie omówiliśmy dwie kluczowe architektury głębokich sieci Transformer stosowanych do wyszukiwania informacji - bi-encoder i cross-encoder.

Przykładem bi-encodera jest `SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')`


```
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("multi-qa-mpnet-base-dot-v1")

docs = [
    "My first paragraph. That contains information",
    "Python is a programming language.",
]
document_embeddings = model.encode(docs)

query = "What is Python?"
query_embedding = model.encode(query)

similarity_scores = embedder.similarity(query_embedding, document_embeddings)

```


Przykładem CrossEncoder (`CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2')`), który w odróżnieniu od bi-encodera ocenia odpowiedniość dokumentu do zapytania, zwracając dla podanej pary tekstów ich wskaźnik dopasowania (prawdopodobieństwo dopasowania).

```
from sentence_transformers import CrossEncoder
import torch

# Load https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2
model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2", activation_fn=torch.nn.Sigmoid())
scores = model.predict([
    ("How many people live in Berlin?", "Berlin had a population of 3,520,031 registered inhabitants in an area of 891.82 square kilometers."),
    ("How many people live in Berlin?", "Berlin is well known for its museums."),
])
```

Do pracy wymagamy zainstalowania trzech pakietów - `sentence-transformers` oraz `rank_bm25` oraz `datasets`


In [1]:
!pip install -U sentence-transformers rank_bm25 datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


Zanim przejdziemy do zdefiniowania zadania przygotujemy sobie kilka przykładów użycia różnych metod i modeli, abyście mieli podstawy do realizacji swojego wyzwania.


Przykładowe użycie leksykalnego wyszukiwania opartego na BM25

In [2]:
from rank_bm25 import BM25Okapi

# Korpus z przykładowymi tekstami
corpus = [
    "Trzeba jeść dużo warzyw, owoców i uprawiać sport.",
    "Trzeba pić alkohol, imprezować i palić.",
    "Kampania polityczna bardzo napawa optymizmem, same mądre głowy chcą rządzić Polakami.",
    "Historia Polski to wiele wieków walki o niepodległość.",
    "Przemysł motoryzacyjny od lat walczy z rosnącymi kosztami produkcji.",
]

tokenized_corpus = [doc.lower().split(" ") for doc in corpus]

bm25 = BM25Okapi(tokenized_corpus)

## KIEDY ZAPYTANIE MA LEKSYKALNE PRZECIĘCIE z KORPUSEM

query_0 = "O czym jest historia Polski?"
print(query_0)

tokenized_query_0 = query_0.lower().split(" ")

#policz dopasowanie leksykalne zapytania do każdego dokumentu w korpusie
doc_scores_0 = bm25.get_scores(tokenized_query_0)
print(doc_scores_0)

#znajdz top N najbliższych dokumentów do zadanego zapytania
print(bm25.get_top_n(tokenized_query_0, corpus, n=1))

## KIEDY ZAPYTANIE NIE MA LEKSYKALNEGO PRZECIĘCIE z KORPUSEM --> Jak długo żyć? nie ma leksykalnych cech wspólnych (tych samych fragmentów słów/tokenów)

query_1 = "Jak długo żyć?"
print(query_1)

tokenized_query_1 = query_1.lower().split(" ")

#policz dopasowanie leksykalne zapytania do każdego dokumentu w korpusie
doc_scores_1 = bm25.get_scores(tokenized_query_1)
print(doc_scores_1)

#znajdz top N najbliższych dokumentów do zadanego zapytania
print(bm25.get_top_n(tokenized_query_1, corpus, n=1))



O czym jest historia Polski?
[0.         0.         0.         2.24533898 0.        ]
['Historia Polski to wiele wieków walki o niepodległość.']
Jak długo żyć?
[0. 0. 0. 0. 0.]
['Przemysł motoryzacyjny od lat walczy z rosnącymi kosztami produkcji.']


Przykładowe użycie semantycznego wyszukiwania bazującego na rozwiazaniach typu bi-encoder dla j.polskiego.

In [4]:
import torch

from sentence_transformers import SentenceTransformer


embedder = SentenceTransformer("sdadas/mmlw-retrieval-roberta-large")

query_prefix = "zapytanie: "
answer_prefix = ""

# Korpus z przykładowymi tekstami
corpus = [
    answer_prefix + "Trzeba jeść dużo warzyw, owoców i uprawiać sport.",
    answer_prefix + "Trzeba pić alkohol, imprezować i palić.",
    answer_prefix + "Kampania polityczna bardzo napawa optymizmem, same mądre głowy chcą rządzić Polakami.",
    answer_prefix + "Historia Polski to wiele wieków walki o niepodległość.",
    answer_prefix + "Przemysł motoryzacyjny od lat walczy z rosnącymi kosztami produkcji.",
]



# Use "convert_to_tensor=True" to keep the tensors on GPU (if available)
corpus_embeddings = embedder.encode(corpus, convert_to_tensor=True)

# Zapytania
queries = [
    query_prefix + "Jak dożyć 100 lat?",
    query_prefix + "Z czym walczy sektor automoto?"
]

# Znajdz dwa najbliższe dokumenty z korpusu dla zadanego zapytania używajać podobieństwa opartego na mierze cosinus
top_k = min(2, len(corpus))
for query in queries:
    query_embedding = embedder.encode(query, convert_to_tensor=True)

    # We use cosine-similarity and torch.topk to find the highest 2 scores
    similarity_scores = embedder.similarity(query_embedding, corpus_embeddings)[0]
    scores, indices = torch.topk(similarity_scores, k=top_k)

    print("\nZapytanie:", query)
    print("Top 2 najbliższych dokumentów:")

    for score, idx in zip(scores, indices):
        print(corpus[idx], f"(Score: {score:.4f})")


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


Zapytanie: zapytanie: Jak dożyć 100 lat?
Top 2 najbliższych dokumentów:
Trzeba jeść dużo warzyw, owoców i uprawiać sport. (Score: 0.7192)
Trzeba pić alkohol, imprezować i palić. (Score: 0.6929)

Zapytanie: zapytanie: Z czym walczy sektor automoto?
Top 2 najbliższych dokumentów:
Przemysł motoryzacyjny od lat walczy z rosnącymi kosztami produkcji. (Score: 0.7715)
Historia Polski to wiele wieków walki o niepodległość. (Score: 0.6724)


- z czym **walczy** sektor auotmoto? - zawiera słowo (/tokeny składające się na słowo) walczy; również top=2 znalezionych dokumentów zawiera słowo wlaczyć (lub jego odmiany) - więc zawierają też podobne tokeny  

Przykładowe użycie semantycznego wyszukiwania w zakresie komponentu re-rankera opartego na polskich cross-encoderach.

In [6]:
from sentence_transformers import CrossEncoder
import torch.nn

query = "Jak dożyć spokojnej starości w dobrej kondycji?"
answers = [
    "Trzeba zdrowo się odżywiać, oraz unikać alkoholu i papierosów",
    "Trzeba pić alkohol, imprezować i chwytać dzień",
    "Życie to tylko ciąg wspomnień i obrazy z przeszłości."
]

model = CrossEncoder(
    "sdadas/polish-reranker-large-ranknet",
    device="cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"
)
pairs = [[query, answer] for answer in answers]
results = model.predict(pairs)
print(results.tolist())


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[0.8108768463134766, 0.3784073293209076, 0.0719650462269783]


ZADANIE #1 PREPROCESSING - przetwórz korpus polskiej wikipedii do postaci kolekcji tekstów

In [1]:
import json
import gzip
import os
import torch

def prep_passages(dataset):
    used_passages = set()
    passages = []
    for doc in dataset[::3]:
        if doc["title"] not in used_passages:
           passages.append(doc["text"])
           used_passages.add(doc["title"])
    return passages

if not torch.cuda.is_available():
    print("Warning: No GPU found. Please add GPU to your notebook")

# Użyj dumpa polskiej Wikipedii.

wikipedia_hf_path = 'chrisociepa/wikipedia-pl-20230401'


from datasets import load_dataset

dataset = load_dataset(wikipedia_hf_path, split="train", streaming=True)

doc = next(iter(dataset))
print(doc)
print(doc.keys())
print(f"Number of docments in dataset: {len(list(dataset))}") # 1 562 327 dokumentów

##Napisz funkcję, która zbuduje min kilka set tys zbiór dokumentów zwanych pasażami
##Używając tej funkcji wypełnij zmienną passages, sprawdz i usun duplikaty

passages = prep_passages(dataset)
print("Passages:", len(passages))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


{'id': '2', 'url': 'https://pl.wikipedia.org/wiki/AWK', 'title': 'AWK', 'text': 'AWK – interpretowany język programowania, którego główną funkcją jest wyszukiwanie i przetwarzanie wzorców w plikach lub strumieniach danych. Jest także nazwą programu początkowo dostępnego dla systemów operacyjnych będących pochodnymi UNIX-a, obecnie także na inne platformy.\n\nAWK jest językiem, który w znacznym stopniu wykorzystuje tablice asocjacyjne, stringi i wyrażenia regularne. Nazwa języka pochodzi od pierwszych liter nazwisk jego autorów Alfreda V. Aho, Petera Weinbergera i Briana Kernighana. Bywa zapisywana małymi literami, odczytywana jako jedno słowo awk, wymawiana jak pierwsza sylaba w awkward.\n\nDefinicja języka AWK jest zawarta w POSIX 1003.2 Command Language And Utilities Standard. Wersja ta jest z kolei oparta na opisie z The AWK Programming Language napisanym przez Aho, Weinbergera i Kernighana, z dodatkowymi właściwościami zdefiniowanymi w wersji awk z SVR4.\n\nW wierszu poleceń podaje

KeyboardInterrupt: 

ZADANIE #2 PRZYGOTOWANIE KOMPONENTÓW - BI-ENCODERA, CROSS-ENCODERA i BM25

In [ ]:
!pip install nltk

In [ ]:

#inicjalizacja Bi-Encodera, dokładniej enkodera używanego w obu gałęziach, aby zbudować embeddingi dla tekstów i zapytania
#sam wybierz dokładny model pod kątem j.polskiego
bi_encoder = SentenceTransformer("sdadas/mmlw-retrieval-roberta-large")

#inicjalizacja Cross-encoder do  re-rankowania wyników
#sam wybierz dokładny model pod kątem j.polskiego
cross_encoder = CrossEncoder(
      "sdadas/polish-reranker-large-ranknet",
      device="cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"
    )

# lexical search (keyword search) z użyciem BM25 metody

from rank_bm25 import BM25Okapi
import string
from tqdm import tqdm
from nltk.corpus import stopwords


# Tokenizacja pod BM25 z uwzglednieniem lower case, usuwaniem stop-words
def bm25_tokenizer(text):
    tokenized_doc = text.lower().replace('\n\n', ' ').replace('\n', ' ').split(' ')
    tokenized_doc = [token for token in tokenized_doc if token not in stopwords.words('polish')]
    return tokenized_doc


tokenized_corpus = []
for passage in tqdm(passages):
    tokenized_corpus.append(bm25_tokenizer(passage))

bm25 = BM25Okapi(tokenized_corpus)

corpus_embeddings = bi_encoder.encode(passages, convert_to_tensor=True, show_progress_bar=True)



ZADANIE #3 - FINALNE ROZWIAZANIE - napisz funkcje, która będzie używała BM25 i BI_ENCODERA do wyszukiwania, potem wyniki z obu tych modułów wyszukiwania należy zmergować i dokonać ich re-rankingu i zwrócić top 10 wyników.

In [ ]:
# Funkcja, która dla tekstowego zapytania wykona Hybrid Search (BM25, BI-ENCODER), potem dokona RE-RANKINGU i zwróci top 10
def search(query, top_n=10):
    print("Input question:", query)

    ##### BM25 search (lexical search) #####
    tokenized_query = query.lower().split(" ")
    doc_scores = bm25.get_scores(tokenized_query)
    print(f"BM25 scores for document corpus/passages: {doc_scores}")

    #znajdz top N najbliższych dokumentów do zadanego zapytania
    bm25_retrived_docs = bm25.get_top_n(tokenized_query, passages, n=top_n)
    print(f"Top={top_n} BM25 search results: {bm25.get_top_n(tokenized_query, passages, n=top_n)}")

    ##### Semantic Search using BI-ENCODER #####
    # Encode the query using the bi-encoder and find potentially relevant passages
    query_embedding = bi_encoder.encode(query, convert_to_tensor=True)
    similarity_scores = bi_encoder.similarity(query_embedding, corpus_embeddings)[0]    # similarity_scores for one query -> [[.., .., ..]] -> inexing similarity_scores[0] gives -> [.., .., ..]
    scores, indices = torch.topk(similarity_scores, k=top_n)
    bi_encoder_retrived_passages = []
    for idx in indices:
        bi_encoder_retrived_passages.append(passages[idx])

    #### MERGE RESULTS FROM BM25 and BI-ENCODER####
    merged_bm25_bi_encoder_res = bm25_retrived_docs + bi_encoder_retrived_passages

    ##### Re-Ranking #####
    # Now, score all retrieved passages with the cross_encoder
    pairs = [[query, doc] for doc in merged_bm25_bi_encoder_res]
    results = cross_encoder.predict(pairs)

    # RETRIEVE TOP N
    # first processing with bm25 + bi-encoder (faster/less demanding methods) + further retrive with cross-encoder on smaller (already 'filtered') text set (as cross-encoder is more computationally demanding)
    print(f"Top={top_n} docs retrived by cross-encoder from bm25 + bi-encoder results merged") 
    scores, indices = torch.topk(results, k=top_n)
    for score, idx in zip(scores, indices):
        print(passage[idx], f"(Score: {score:.4f})")


In [ ]:
search(query = "Ile mieszkańców ma Warszawa?")

In [ ]:
search(query = "Jak długa jest Wisła?")

In [ ]:
search(query = "Podaj pięciu polskich Noblistów?")

In [ ]:
search(query = "Jak nazywał się pierwszy król Polski?")